# Dự báo Doanh thu Bán lẻ
## 8. Kết luận & Định hướng Phát triển Dự án

Báo cáo kết luận này tổng kết toàn bộ quá trình thực nghiệm, từ bài toán dự báo doanh thu cấp độ hóa đơn đơn lẻ cho đến dự báo chuỗi thời gian doanh thu gộp tuần của các siêu thị/ngành hàng. Dưới đây là những nhìn nhận khách quan, thực tế về hiệu năng mô hình cùng các khuyến nghị cải tiến phục vụ hoạt động vận hành thực tế.

---

### 8.1. Tổng kết Hiệu năng Thực nghiệm

Dự án đã triển khai và so sánh thành công hai cấp độ dự báo bằng lớp thuật toán tự viết **LightGBM (Scratch)** đối chứng với các thư viện Boosting hàng đầu hiện nay (**HistGradientBoosting, XGBoost, LightGBM, CatBoost**):

1. **Bài toán Dự báo cấp độ Hóa đơn (Tabular Data):**
   * **Hiệu năng:** Các mô hình đạt hệ số xác định $R^2 \approx 0.487$.
   * **Đánh giá thực tế:** Để triệt tiêu rủi ro rò rỉ dữ liệu (target leakage), chúng ta đã loại bỏ hai trường quyết định doanh thu trực tiếp là `quantity` và `price`. Việc mô hình chỉ dựa vào các biến phi cấu trúc gián tiếp (như độ tuổi, giới tính, trung tâm thương mại, ngành hàng) vẫn giải thích được gần 50% phương sai là một kết quả thực tế, phản ánh đúng tính ngẫu nhiên cao trong hành vi mua sắm của từng khách hàng cá nhân.
   * **Kiểm nghiệm Scratch:** Mô hình LightGBM Scratch cho hiệu năng tương đương với XGBoost và CatBoost, xác nhận logic phân tách cây và tính gradient tự viết hoạt động chính xác.

2. **Bài toán Dự báo cấp độ Chuỗi thời gian Tuần (Time-Series Aggregation):**
   * **Hiệu năng (mô hình Scratch):** Đạt $R^2 \approx 87.0\%$, sai số tuyệt đối trung bình $MAE \approx 2035.06$ TRY, sai số bình phương trung bình $RMSE \approx 4504.85$ TRY.
   * **Đánh giá thực tế:** 
     * Mức $R^2$ cao ($87.0\%$) phản ánh các đặc trưng tự hồi quy (Lag, Rolling) đã nắm bắt tốt xu hướng mùa vụ (seasonality) và tính tự tương quan (autocorrelation) của chuỗi doanh số bán lẻ.
     * Tuy nhiên, mức sai số tuyệt đối trung bình **MAE chiếm tới 26.6%** doanh thu tuần trung bình. Trong vận hành kho vận, đây là mức sai số **khá lớn**, đòi hỏi doanh nghiệp phải duy trì một lượng tồn kho an toàn (safety stock) lớn để bù đắp rủi ro thiếu hàng.
     * Sai số **RMSE cao gấp hơn 2 lần MAE** chỉ ra mô hình có độ biến động sai số lớn và kém ổn định trước các điểm nhu cầu đột biến (peak demand) hoặc các tuần lễ hội/khuyến mại lớn.


### 8.2. Các Hạn chế của Mô hình Hiện tại

Để ứng dụng mô hình vào thực tế doanh nghiệp, cần nghiêm túc nhìn nhận các điểm nghẽn kỹ thuật sau:
1. **Thiếu thông tin ngoại cảnh tác động:** Mô hình hiện tại chỉ khai thác các thuộc tính nội bộ của giao dịch (lịch sử doanh thu, ngành hàng, địa điểm). Doanh thu bán lẻ thực tế phụ thuộc rất mạnh vào: các chương trình khuyến mãi (promotions), lịch sự kiện/ngày lễ quốc gia, thời tiết, và các chỉ số kinh tế vĩ mô (lạm phát, tỷ giá).
2. **Độ nhạy cao với Outliers:** Việc RMSE lớn chứng minh mô hình dễ bị lệch pha nghiêm trọng khi nhu cầu thị trường biến động phi tuyến tính.
3. **Hiệu năng tính toán của thuật toán Scratch:** Phiên bản tự viết dù đạt độ chính xác tương đương thư viện chuẩn nhưng chưa được tối ưu hóa chia giỏ (Histogram binning) và song song hóa cấp độ luồng (C++ backend), dẫn đến thời gian huấn luyện tăng nhanh khi kích thước dữ liệu mở rộng.


### 8.3. Khuyến nghị & Định hướng Tối ưu hóa

Để nâng cao chất lượng dự báo lên mức có thể đưa vào vận hành thực tế (Production-ready), nhóm nghiên cứu đề xuất các định hướng tối ưu hóa sau:

#### 1. Cải tiến Tiền xử lý & Kỹ thuật Đặc trưng (Feature Engineering)
* **Biến đổi Logarit biến mục tiêu:** Áp dụng phép biến đổi $y_{\text{new}} = \log(y + 1)$ trước khi đưa vào huấn luyện mô hình hồi quy. Do doanh thu bán lẻ thường có phân phối lệch phải (right-skewed) với nhiều hóa đơn giá trị cực lớn, biến đổi logarit sẽ giúp thu hẹp khoảng cách sai số bình phương và làm giảm RMSE.
* **Bổ sung các đặc trưng lịch sự kiện:** Tạo các biến giả (dummy variables) đánh dấu các tuần lễ hội, kỳ nghỉ hè, ngày Black Friday hoặc các tuần chạy chiến dịch marketing.
* **Phân cụm chuỗi thời gian (Time-series Clustering):** Gom nhóm các cửa hàng hoặc ngành hàng có chung mô hình hành vi mua sắm (ví dụ: nhóm nhu cầu cao vào cuối tuần vs nhóm ổn định ngày thường) để xây dựng các mô hình dự báo chuyên biệt.

#### 2. Cải tiến Mô hình hóa & Cân chỉnh Tham số
* **Tận dụng API Categorical gốc của Thư viện:** Chuyển sang sử dụng trực tiếp tính năng xử lý categorical của LightGBM/CatBoost thay vì One-Hot Encoding để tránh hiện tượng ma trận thưa và tăng tốc độ hội tụ của cây.
* **Tối ưu hóa Hyperparameter tự động:** Sử dụng các framework tối ưu hóa như **Optuna** (Bayesian Optimization) để tìm kiếm các tham số tối ưu cho mô hình (`max_depth`, `learning_rate`, `num_leaves`, `reg_alpha`, `reg_lambda`), đặc biệt là việc bổ sung regularization L1/L2 để hạn chế overfitting và giảm RMSE.

#### 3. Ứng dụng trong Vận hành Thực tế (Business Integration)
* **Thiết lập mức Tồn kho An toàn (Safety Stock):** Kết quả MAE ~26.6% cần được tích hợp trực tiếp vào công thức tính điểm đặt hàng lại (Reorder Point). Lượng hàng dự trữ an toàn nên được tính bằng công thức: 
  $$\text{Safety Stock} = Z \times \sqrt{\text{Lead Time}} \times \text{RMSE}$$
  để đảm bảo giảm thiểu tối đa tỷ lệ đứt gãy nguồn cung ngay cả trong các tuần mô hình dự báo bị lệch nhiều nhất.
